[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/notebooks/01_the_idea_of_learning.ipynb)

# ⚒️ Act I · Quest 01 — The Idea of Learning

*Before any framework: what IS learning? Knobs, loss, and rolling downhill — in pure Python.*

[02_build_your_own_autograd](02_build_your_own_autograd.ipynb) ➡️

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, random

np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## 🎮 A game: guess the machine

I'm thinking of a machine that turns numbers into numbers: `y = w * x`. You don't know `w`.
You only get to see examples. Your job — and the job of *every* neural network ever trained —
is to find the knob setting `w` that best explains the data.

In [ ]:
SECRET_W = 3.7   # 🤫 pretend you can't see this

xs = [1.0, 2.0, 3.0, 4.0, 5.0]
ys = [SECRET_W * x for x in xs]
print("examples the machine produced:", list(zip(xs, ys)))

### Step 1 — measure how wrong a guess is: the **loss**

Pick any guess for `w`. The **loss** is a single number scoring how badly it explains the data.
We'll use mean squared error: average of `(prediction − truth)²`.

In [ ]:
def loss(w):
    return sum((w * x - y) ** 2 for x, y in zip(xs, ys)) / len(xs)

for guess in [0.0, 2.0, 3.7, 6.0]:
    print(f"w={guess:>4}: loss = {loss(guess):8.3f}")

In [ ]:
ws = [i / 10 for i in range(-20, 100)]
plt.plot(ws, [loss(w) for w in ws])
plt.axvline(3.7, ls="--", c="green", label="secret w")
plt.xlabel("guess for w"); plt.ylabel("loss"); plt.legend()
plt.title("The loss is a valley. Learning = finding the bottom."); plt.show()

### Step 2 — which way is downhill? the **gradient**

We could try every `w`... but a real network has *millions* of knobs. Instead, ask the slope:
nudge `w` a tiny bit and see how the loss reacts. That's a **finite-difference derivative**:

\[ \frac{dL}{dw} \approx \frac{L(w+h) - L(w-h)}{2h} \]

Positive slope → downhill is to the *left*. Negative slope → downhill is to the *right*.

In [ ]:
def slope(f, w, h=1e-5):
    return (f(w + h) - f(w - h)) / (2 * h)

print("slope at w=1.0:", slope(loss, 1.0), " -> negative, so move RIGHT (increase w)")
print("slope at w=6.0:", slope(loss, 6.0), " -> positive, so move LEFT (decrease w)")
print("slope at w=3.7:", slope(loss, 3.7), " -> ~0, we'd be at the bottom")

### Step 3 — roll downhill: **gradient descent**

Repeat: measure the slope, take a small step *against* it. The step size is the
**learning rate** — the single most important dial in deep learning.

In [ ]:
w = 0.0                      # start with a terrible guess
lr = 0.01                    # learning rate
history = [w]
for step in range(40):
    w = w - lr * slope(loss, w)
    history.append(w)

print(f"final guess: w = {w:.4f}   (secret was {SECRET_W})")
plt.plot(history, marker=".")
plt.axhline(SECRET_W, ls="--", c="green")
plt.xlabel("step"); plt.ylabel("w"); plt.title("Rolling downhill to the answer"); plt.show()

### Step 4 — more knobs, same trick

Real models have many knobs. Here's `y = w*x + b` — two knobs. The recipe doesn't change:
compute the slope **for each knob**, step each one downhill. (The collection of all slopes is
called *the gradient*.)

In [ ]:
SECRET = (2.5, -1.0)          # secret w, b
data = [(x, SECRET[0] * x + SECRET[1]) for x in xs]

def loss2(w, b):
    return sum((w * x + b - y) ** 2 for x, y in data) / len(data)

w, b = 0.0, 0.0
for step in range(2000):
    dw = slope(lambda v: loss2(v, b), w)
    db = slope(lambda v: loss2(w, v), b)
    w, b = w - 0.01 * dw, b - 0.01 * db

print(f"learned: w={w:.3f}, b={b:.3f}   (secret {SECRET})")

## 🧭 The universal recipe

Everything in this course — CNNs, GPTs, GANs — is this exact loop with fancier machines:

```
for each step:
    prediction = machine(inputs)                 # forward
    loss       = how_wrong(prediction, truth)    # measure
    gradient   = slope of loss w.r.t. EVERY knob # backward
    knobs     -= learning_rate * gradient        # descend
```

One problem: finite differences need **2 loss evaluations per knob per step**. GPT-4-class
models have ~10¹² knobs. We need something smarter than nudging — and in the next quest,
**you will build it**: an engine that computes *all* the slopes in one backward sweep.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda ans: abs(ans - 42) < 1e-9,
    "the answer to everything")
_register("grad_fn", 15,
    lambda f: abs(f(lambda w: (w - 2) ** 2, 5.0) - 6.0) < 1e-3,
    "return (f(w+h) - f(w-h)) / (2*h) with a small h like 1e-5")
_register("descend", 15,
    lambda w: abs(w - 4.0) < 0.05,
    "run gradient descent on g(w) = (w-4)**2, ~100 steps with lr=0.1, starting anywhere")
_register("lr_quiz", 10,
    lambda s: s.strip().lower() == "diverge",
    "one word: what does the loss do if the learning rate is far too big — 'converge', 'diverge', or 'freeze'?")

In [ ]:
# warm-up: the grader in action. Run me!
check("warmup", 42)

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `grad_fn` (15 XP) — write `my_slope(f, w)` that estimates df/dw by finite differences, then `check("grad_fn", my_slope)`.
- `descend` (15 XP) — minimize `g(w) = (w - 4)**2` with your own descent loop; `check("descend", final_w)`.
- `lr_quiz` (10 XP) — `check("lr_quiz", "your one-word answer")`.

In [ ]:
# ⚔️ your attempts here...
# def my_slope(f, w, h=1e-5): ...
# check("grad_fn", my_slope)

# xp_report()